<a href="https://colab.research.google.com/github/Diana-Tafy/-EXERCISES/blob/main/Patient_Readmission.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
from google.colab import drive
import pandas as pd
import numpy as np

# Mount Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [65]:
data_path='/content/drive/MyDrive/Patient Readmission/Patient_Readmission csv.xlsx'

df=pd.read_excel(data_path)

#returns number of rows
df.head(5)



,Patient ID,Age,Gender,Admission Type,Length of Stay,Number of Diagnoses,Blood Pressure,Blood Sugar Levels,Previous Admissions,Readmission
0,1,62,Female,Elective,4,5,110,130,1,No
1,2,65,Male,Emergency,19,2,157,81,4,No
2,3,82,Female,Emergency,18,4,74,84,0,No
3,4,85,Male,Emergency,2,4,106,85,4,No
4,5,85,Female,Elective,19,3,80,119,3,No


**SUBJECT: Advanced Data Analysis & Key Metrics for Patient Readmission Dataset**

I am analyzing a hospital patient readmission dataset to extract clinical and operational insights. To accomplish this, I need to translate standard SQL data-manipulation workflows into Python using the pandas library.


1.WHERE Statement: Filtering rows based on specific conditional criteria (e.g., patient demographics or clinical thresholds).

2.DISTINCT: Identifying unique values or eliminating duplicate records across specific categories.

3.GROUP BY: Segmenting the dataset by categorical fields to prepare for aggregation.

4.HAVING: Filtering aggregated summary groups after a groupby operation has been executed.

5.AVG: Calculating mathematical averages (means) across continuous numerical variables.

6.LOGICAL OPERATORS: Combining multiple search conditions using complex logic like AND (&), OR (|), and NOT (~).

7.ANY & ALL OPERATORS: Evaluative filtering to check if any or all values in a specific series meet a given condition.

8.WILDCARDS: Performing string pattern-matching and text searches within categorical variables.

In [23]:
df.columns

Index(['Patient ID', 'Age', 'Gender', 'Admission Type', 'Length of Stay',
       'Number of Diagnoses', 'Blood Pressure', 'Blood Sugar Levels',
       'Previous Admissions', 'Readmission'],
      dtype='object')

**Observation**: In the WHERE statement, we can identify the number of patients who are 'frequent flyers' with more than 3 previous admissions.

In [30]:
#1. THE 'WHERE' STATEMENT (Filtering)

# Filter for patients with more than 3 previous admissions

frequent_flyers = df[df['Previous Admissions'] > 3]

In [31]:
frequent_flyers.head()

,Patient ID,Age,Gender,Admission Type,Length of Stay,Number of Diagnoses,Blood Pressure,Blood Sugar Levels,Previous Admissions,Readmission
1,2,65,Male,Emergency,19,2,157,81,4,No
3,4,85,Male,Emergency,2,4,106,85,4,No
16,17,27,Male,Elective,22,1,118,93,4,No
22,23,47,Male,Emergency,7,9,118,56,4,Yes
23,24,37,Male,Emergency,8,9,100,105,4,No


Observations :The total number of patients found is 561

In [37]:
print("Number of patients found:", len(frequent_flyers))

Number of patients found: 561


**Observation:** Using the DISTINCT logic on the Admission Type column shows that patients in this dataset enter the facility through exactly two primary channels: 'Elective' and 'Emergency'.

In [41]:
# Find distinct values in the 'Admission Type' column
distinct_admission_type = df['Admission Type'].unique()
print(distinct_admission_type)

['Elective' 'Emergency']


**Observation**: Clinical Averages by Readmission Status
**Length of Stay**: Patients who were readmitted stayed slightly longer on average (15.50 days) than those who were not (15.11 days).

**Blood Sugar Levels**: Both groups show elevated average blood sugar levels, with readmitted patients marginally higher (134.52 mg/dL) than non-readmitted patients (133.14 mg/dL).

I used two different syntax to see if l get the same outcome

In [48]:
# Group by Readmission and calculate the mean for both columns
df.groupby('Readmission')[['Length of Stay', 'Blood Sugar Levels']].mean()

,Length of Stay,Blood Sugar Levels
Readmission,,
No,15.113871,133.136364
Yes,15.503464,134.520785


In [51]:
# Group by Readmission status ('Yes'/'No') and find the average length of stay and blood sugar
avg_stats_by_readmit = df.groupby('Readmission').agg(
    Avg_Length_of_Stay=('Length of Stay', 'mean'),
    Avg_Blood_Sugar=('Blood Sugar Levels', 'mean')
)

In [53]:
#Outcome of the above code
avg_stats_by_readmit

,Avg_Length_of_Stay,Avg_Blood_Sugar
Readmission,,
No,15.113871,133.136364
Yes,15.503464,134.520785


**Observation**:Both admission channels easily cleared the filter threshold of 2, with Elective admissions averaging 4.97 diagnoses and Emergency admissions averaging 4.94 diagnoses.

In [63]:
#HAVING (Filtering Aggregated Data)

# Find admission types where the average number of diagnoses is greater than 2

# Step 1: Group and aggregate
admission_summary = df.groupby('Admission Type').agg(
    Avg_Diagnoses=('Number of Diagnoses', 'mean')
)
# Step 2: Apply the 'HAVING' filter
high_risk_admission_types = admission_summary[admission_summary['Avg_Diagnoses'] > 2]

In [64]:

#Outcome from the above code
high_risk_admission_types

,Avg_Diagnoses
Admission Type,
Elective,4.972489
Emergency,4.939457


**Observations**:
**Emergency Readmissions** (AND): Readmitted emergency patients show high clinical complexity, with most of the top rows presenting 8 to 9 distinct diagnoses.

**Metabolic Risk Profile** (OR): The metabolic filter was heavily triggered by high blood pressure (e.g., values >135) rather than high blood sugar, though many in this high-risk subset were safely discharged without a readmission.

**Non-Elective Admissions** (NOT): Removing elective cases isolates an unpredictable emergency workload, showing a wide variance in initial hospital stays ranging anywhere from 2 to 19 days.

In [66]:
#LOGICAL OPERATORS (AND, OR, NOT)

# AND (&): Emergency patients who were ultimately readmitted
emergency_readmit = df[(df['Admission Type'] == 'Emergency') & (df['Readmission'] == 'Yes')]

# OR (|): Patients who have high blood pressure OR high blood sugar
metabolic_risk = df[(df['Blood Pressure'] > 130) | (df['Blood Sugar Levels'] > 140)]

# NOT (~): All patients who did NOT enter via 'Elective' admission
non_elective = df[~(df['Admission Type'] == 'Elective')]



In [68]:
print("--- Emergency Readmissions ---")
display(emergency_readmit.head())

print("\n--- Metabolic Risk Patients ---")
display(metabolic_risk.head())

print("\n--- Non-Elective Admissions ---")
display(non_elective.head())

--- Emergency Readmissions ---


,Patient ID,Age,Gender,Admission Type,Length of Stay,Number of Diagnoses,Blood Pressure,Blood Sugar Levels,Previous Admissions,Readmission
11,12,83,Male,Emergency,13,8,110,78,2,Yes
15,16,43,Male,Emergency,28,1,128,100,0,Yes
22,23,47,Male,Emergency,7,9,118,56,4,Yes
24,25,37,Male,Emergency,19,8,124,108,1,Yes
29,30,27,Male,Emergency,10,9,151,138,4,Yes



--- Metabolic Risk Patients ---


,Patient ID,Age,Gender,Admission Type,Length of Stay,Number of Diagnoses,Blood Pressure,Blood Sugar Levels,Previous Admissions,Readmission
1,2,65,Male,Emergency,19,2,157,81,4,No
5,6,27,Male,Emergency,18,6,136,99,2,No
7,8,54,Male,Emergency,3,4,136,68,3,No
9,10,30,Male,Elective,6,8,139,106,3,No
17,18,38,Male,Elective,8,4,147,116,3,Yes



--- Non-Elective Admissions ---


,Patient ID,Age,Gender,Admission Type,Length of Stay,Number of Diagnoses,Blood Pressure,Blood Sugar Levels,Previous Admissions,Readmission
1,2,65,Male,Emergency,19,2,157,81,4,No
2,3,82,Female,Emergency,18,4,74,84,0,No
3,4,85,Male,Emergency,2,4,106,85,4,No
5,6,27,Male,Emergency,18,6,136,99,2,No
7,8,54,Male,Emergency,3,4,136,68,3,No


**Observations**:
 (**any)**: Returning True confirms that long-stay outliers (stays > 20 days) exist in our population, representing high-resource cases.

 (**all**): Returning True validates perfect data integrity, confirming every single patient record has at least 1 primary diagnosis coded.

In [71]:
#ANY & ALL OPERATORS

# To check if ANY patient has a length of stay longer than 20 days (Returns True/False)
has_long_stay = (df['Length of Stay'] > 20).any()

# To check if ALL patients have at least 1 diagnosis recorded (Returns True/False)
all_have_diagnoses = (df['Number of Diagnoses'] >= 1).all()

# Print the text labels as strings
print("Does any patient stay longer than 20 days?")
print(has_long_stay)

print("\nDo all patients have at least 1 diagnosis?")
print(all_have_diagnoses)

Does any patient stay longer than 20 days?
True

Do all patients have at least 1 diagnosis?
True


In [75]:
#WILDCARDS (Text Pattern Matching)

# If 'Admission Type' or any notes column contains text patterns (e.g., matching 'Emerg')
emergency_like = df[df['Admission Type'].str.contains('Emerg', case=False, na=False)]

print(emergency_like.head())

   Patient ID  Age  Gender Admission Type  Length of Stay  \
1           2   65    Male      Emergency              19   
2           3   82  Female      Emergency              18   
3           4   85    Male      Emergency               2   
5           6   27    Male      Emergency              18   
7           8   54    Male      Emergency               3   

   Number of Diagnoses  Blood Pressure  Blood Sugar Levels  \
1                    2             157                  81   
2                    4              74                  84   
3                    4             106                  85   
5                    6             136                  99   
7                    4             136                  68   

   Previous Admissions Readmission  
1                    4          No  
2                    0          No  
3                    4          No  
5                    2          No  
7                    3          No  
